In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers

In [ ]:
import torch
import math
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, set_seed
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
SEED = 42
BATCH_SIZE = 32
MAX_LENGTH = 512
STRIDE = 128
MLM_PROBABILITY = 0.15

In [ ]:
MODELS = [
    "dbmdz/bert-base-italian-xxl-cased",
    "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]
DATASET_PATH = "/content/drive/My Drive/italian_poems/italian_poems_test.json"

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fix_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

for model_id in MODELS:
    print(f"\nEvaluating: {model_id}")
    fix_seed(SEED)

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    print(type(tokenizer))
    model = AutoModelForMaskedLM.from_pretrained(model_id).to(device)
    model.eval()

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            stride=STRIDE,
            return_overflowing_tokens=True,
            return_special_tokens_mask=True
        )

    tokenized_datasets = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing"
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROBABILITY
    )

    dataloader = DataLoader(
        tokenized_datasets,
        batch_size=BATCH_SIZE,
        collate_fn=data_collator
    )

    batch_losses = []

    for batch in tqdm(dataloader, desc="Calculating Pseudo-PPL"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss
        batch_losses.append(loss.item())

    losses_np = np.array(batch_losses)
    batch_ppls = np.exp(losses_np)
    mean_loss = losses_np.mean()
    corpus_perplexity = math.exp(mean_loss)

    std_ppl = np.std(batch_ppls)
    q25 = np.percentile(batch_ppls, 25)
    q75 = np.percentile(batch_ppls, 75)
    iqr_ppl = q75 - q25

    print(f"\nResults for {model_id}:")
    print(f"--------------------------------------------------")
    print(f"Average Loss:         {mean_loss:.4f}")
    print(f"Corpus Perplexity:    {corpus_perplexity:.2f}")
    print(f"Perplexity Std Dev:   {std_ppl:.2f}")
    print(f"Perplexity IQR:       {iqr_ppl:.2f}  [Q1: {q25:.2f}, Q3: {q75:.2f}]")
    print(f"--------------------------------------------------")

    del model
    torch.cuda.empty_cache()